In [16]:
import pandas as pd
# ---- run me first in every notebook ----
import sys, os
from pathlib import Path

# Ensure we are at the project root (the folder that contains `src/`)
# If your notebook sits in the root, this is already correct.
root = Path.cwd()

print("Project root:", root)

# Get the directory of the current script
current_dir = os.getcwd()
# Get the parent directory
parent_dir = os.path.dirname(current_dir)
grandparent_dir = os.path.dirname(parent_dir)

# Add the parent directory to the system path
sys.path.append(parent_dir)
sys.path.append(grandparent_dir)
print(grandparent_dir)

Project root: d:\Bioinformatics\Rosaloid\data\round1
d:\Bioinformatics\Rosaloid


In [ ]:
# round1_prep_train_gp.py  — build per-method training matrices from .npy cache
from pathlib import Path
import pandas as pd, numpy as np, json
from src.esm_feats import load_esm1v, embed_dataframe  # assuming esm_feats.py is in the src directory

METHODS = ["additive","plldelta","mutant_ctx"]

# where your round0 labeled files live (adjust if needed)
ROUND0 = Path(Path.cwd().parent / "round0")  # e.g., <proj>/data/round0 if you're running in data/round1

DEVICE = "cuda"  # or "cpu" if no GPU

def robust_norm(y):
    mu = np.median(y); mad = np.median(np.abs(y - mu)) + 1e-9
    return (y - mu)/mad, float(mu), float(mad)

def embed_matrix_for_sequences(seqs):
    """Return NxD embedding matrix aligned to 'seqs' order using your esm_feats cache."""
    load_esm1v(device=DEVICE)  # warm the model once; get_embedding is cached
    meta = embed_dataframe(pd.DataFrame({"mutated_sequence": seqs}))
    paths = meta["embedding_path"].unique()
    if len(paths) != 1:
        raise RuntimeError(f"Expected a single embedding matrix path, got {len(paths)}: {paths}")
    E = np.load(paths[0])  # shape [N, D], aligned to rows of 'meta' (which matches 'seqs')
    if E.shape[0] != len(seqs):
        raise RuntimeError(f"Embedding rows {E.shape[0]} != number of sequences {len(seqs)}")
    return E

for tag in METHODS:
    lab_path = ROUND0 / f"round0_{tag}_labeled.csv"
    if not lab_path.exists():
        print(f"[{tag}] missing {lab_path} — skipping")
        continue

    lab = pd.read_csv(lab_path)

    # training rows only (no controls, no NaNs)
    m = (lab["is_WT"]==0) & (lab["is_blank"]==0) & lab["measured_score"].notna()
    train = lab.loc[m].copy()
    if train.empty:
        print(f"[{tag}] no training rows — skipping")
        continue

    seqs = train["mutated_sequence"].astype(str).tolist()
    X = embed_matrix_for_sequences(seqs)              # <— from .npy cache
    y = train["measured_score"].to_numpy(float)
    y_norm, mu, mad = robust_norm(y)

    # Save a compact NPZ “train matrix”
    out_npz = ROUND0 / f"round0_{tag}_train_matrix.npz"
    np.savez_compressed(out_npz,
        mutated_sequence=np.array(seqs, dtype=object),
        X=X.astype(np.float32, copy=False),
        y_norm=y_norm.astype(np.float32, copy=False)
    )

    # Save normalization params for this method
    Path("cache/models").mkdir(parents=True, exist_ok=True)
    with open(f"cache/models/round0_{tag}_norm.json","w") as f:
        json.dump({"median":mu, "mad":mad}, f, indent=2)

    print(f"[{tag}] n={len(seqs)} | d={X.shape[1]} | wrote {out_npz.name} and norm.json")


C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\esm\pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(


[additive] n=88 | d=1280 | wrote round0_additive_train_matrix.npz and norm.json
[plldelta] n=88 | d=1280 | wrote round0_plldelta_train_matrix.npz and norm.json
[mutant_ctx] n=88 | d=1280 | wrote round0_mutant_ctx_train_matrix.npz and norm.json
